# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/widadfatimakhan/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane 4 — CTR / Engagement Opportunity Scoring.** The data contract for my lane against the full
warehouse release (`FlyRank/internship-warehouse`), with every sentence checked by a query.

Week 3's session walked the river the data travels: a search happens → Search Console and Analytics
record it → native exports land in BigQuery → a nightly `MERGE` aggregates the raw rows to *one
honest row per article per day* → facts + dimensions → the gated, pseudonymized release my notebook
reads. This describes **my slice of that river**.

| Card asks for | Lives in |
|---|---|
| The contract in plain words — 5 answers | §1 + §2 |
| Exactly three verification queries, mid-panel month | §3 — **Q1** grain · **Q2** count + span · **Q3** availability with `IS TRUE` |
| Five features, each "knowable at the decision moment because…" | §3.5 |
| The deliberate leak, shown then removed | §3.6 |
| One named limitation of my slice | §4 |

**Two rules I hold myself to.** *Doubt a column name until you've verified it* — §0 resolves every
name against the live schema. *Two kinds of nothing* — `0` = we looked and found nothing, `NULL` =
we could not look, so every availability filter uses `IS TRUE` / `IS NOT TRUE`, never `= TRUE`.

**Iteration rule.** I develop on `month=2026-03`. The `_sample` table is not a random sample — it is
the panel's final month (June 2026), the natural outcome window of any past→future label. Sealed.

## 0. Setup — connect, and verify every column name before using it

Nothing here is part of the contract yet. This section only makes sure I am pointed at the right
release and that the columns I am about to write sentences about actually exist, under the names
I think they have.

In [79]:
%pip -q install duckdb

# Token: env var -> Colab Secret -> prompt. NEVER paste a token into a cell (this repo is public).
import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
print("token loaded:", bool(HF_TOKEN))

token loaded: True


In [80]:
import duckdb, pandas as pd, numpy as np
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 50)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL   = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"                       # mid-panel development month (NOT the final month)
T = {
    "dim_clients":  f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":  f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_month":   f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')",
    "fact_query90": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# COUNT(*) on Parquet reads metadata, not data -- near-free, and it proves I am on the documented build.
published = {"dim_clients": 104, "dim_content": 519_606, "fact_daily": 78_835_655, "fact_query90": 2_414_248}
for name, want in published.items():
    n = con.sql(f"SELECT COUNT(*) FROM {T[name]}").fetchone()[0]
    print(f"{name:14} {n:>12,} rows   <- {'matches docs' if n == want else f'DIFFERS ({want:,})'}")

dim_clients             104 rows   <- matches docs
dim_content         519,606 rows   <- matches docs


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily       78,835,655 rows   <- matches docs
fact_query90      2,414,248 rows   <- matches docs


In [81]:
# --- Column-name handshake --------------------------------------------------
def columns_of(rel): return list(con.sql(f"SELECT * FROM {rel} LIMIT 0").df().columns)
FACT_COLS, CLIENT_COLS = columns_of(T["fact_month"]), columns_of(T["dim_clients"])

def resolve(cands, avail, label):
    for c in cands:
        if c in avail: return c
    raise KeyError(f"None of {cands} found for '{label}'. Available:\n  " + ", ".join(avail))

COL = {
    "date":     resolve(["report_date"],                              FACT_COLS, "report date"),
    "client":   resolve(["client_hash_id", "client_id"],              FACT_COLS, "client id"),
    "content":  resolve(["content_hash_id", "content_id"],            FACT_COLS, "content id"),
    "imp":      resolve(["gsc_impressions", "impressions"],           FACT_COLS, "impressions"),
    "clicks":   resolve(["gsc_clicks", "clicks"],                     FACT_COLS, "clicks"),
    "pos":      resolve(["gsc_avg_position", "avg_position"],         FACT_COLS, "avg position"),
    "gsc_flag": resolve(["gsc_data_available", "client_has_gsc"],     FACT_COLS, "GSC availability flag"),
    "ga4_flag": resolve(["ga4_data_available", "client_has_ga4"],     FACT_COLS, "GA4 availability flag"),
}

# The release carries TWO flag families (client-level client_has_*, row-level *_data_available) and
# a gsc_sum_position column -- position arrives as a SUM on purpose, so it can be averaged correctly.
HAS_CLIENT_FLAGS = {"client_has_gsc", "client_has_ga4"}.issubset(FACT_COLS)
HAS_MONTH_COL    = "month" in FACT_COLS
HAS_SUM_POSITION = "gsc_sum_position" in FACT_COLS

print(f"{len(FACT_COLS)} fact columns; the 8 this contract needs resolve to:")
for k, v in COL.items(): print(f"   {k:9} -> {v}")
print(f"\nalso present -> client-level flags: {HAS_CLIENT_FLAGS} | month: {HAS_MONTH_COL} | "
      f"gsc_sum_position: {HAS_SUM_POSITION}")
print("\nfact columns:\n ", ", ".join(FACT_COLS))
print("\ndim_clients columns:\n ", ", ".join(CLIENT_COLS))

31 fact columns; the 8 this contract needs resolve to:
   date      -> report_date
   client    -> client_hash_id
   content   -> content_hash_id
   imp       -> gsc_impressions
   clicks    -> gsc_clicks
   pos       -> gsc_avg_position
   gsc_flag  -> gsc_data_available
   ga4_flag  -> ga4_data_available

also present -> client-level flags: True | month: True | gsc_sum_position: True

fact columns:
  report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events, month

dim_clients columns:
  client_hash_id, is_active, has_gsc_access, has_ga4_access, access_profile, client_created_date, cli

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**1 · What one row means.** One row = **one page over one calendar month** — one
`(client_hash_id, content_hash_id)` pair aggregated across its daily rows in `month=2026-03`. The
warehouse's grain is finer: one row per *page per day per client* (the row Haris showed the nightly
merge building). My lane rolls those page-days up, because the decision happens once per page: an
editor picks *which pages* to review, not which page-days.

**2 · Which tables.** `fact_content_daily_performance`, partition `month=2026-03` (31 columns) —
everything is built from it. `dim_clients` is read once as **context only**, for per-client history
coverage (§4). `dim_content` and `fact_content_query_90d` are deliberately not joined yet: the query
table's fixed 90-day window overlaps the months I will later use as an outcome window, and joining
before aligning windows is exactly the quiet leak this assignment is about.

**3 · Which time window.** Observation window **2026-03-01 → 2026-03-31**; decision moment
**2026-04-01**, when an editor opens the queue holding only what was known on the 31st. Every feature
must be computable from `report_date <= 2026-03-31`. One feature deliberately looks *inside* the
window (last-14 vs prior-14); none looks past it.

```text
        observation window (features + proxy)        outcome window (NOT used in ML-04)
   |--------------------------------------------|  |------------------------|      |----------|
   2026-03-01                          2026-03-31   2026-04-01     2026-04-30       2026-06 = SEALED
                                                |
                                       decision moment: the queue is ranked here
```

**4 · What I would rank (a proxy, not ground truth).** A continuous **`ctr_gap_pp`**: how many
percentage points below its position-matched peers a page's click-through rate sits.

```text
ctr_pp_i     = 100 * clicks_i / impressions_i
peer_pp_i    = 100 * (SUM(clicks in tier) - clicks_i) / (SUM(impressions in tier) - impressions_i)
ctr_gap_pp_i = peer_pp_i - ctr_pp_i
```

A **rule I authored**, not an outcome anyone observed. It says a page converts visibility into clicks
worse than comparable pages did — not why, and not that a rewrite would fix it. The queue the editor
sees is ordered by `expected_missed_clicks = impressions × ctr_gap_pp / 100`, where impressions is a
**known multiplier applied afterwards**, never a predicted quantity.

Two upgrades on my ML-03 version, closing limitations I flagged there: the peer baseline is
**leave-one-out** (a page is never in its own benchmark), and it is **volume-weighted** (pooled
clicks ÷ pooled impressions), so a 500-impression page no longer moves the benchmark like a
200,000-impression one does.

**5 · One thing I deliberately exclude.** **All 14 GA4-side columns** — `ga4_pageviews`,
`ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `scroll_events`, the
`sessions_*` channel split, and the `sessions_ai` / `ai_*` family — for this contract, not forever.
Reason: GA4 availability is *three-valued* (TRUE / FALSE / NULL). If availability tracks the
**client**, then any GA4 feature partly encodes which client a page belongs to — precisely what a
client-holdout split exists to hold out. The release ships two different flags (`client_has_ga4` and
`ga4_data_available`) that ask different questions, so **Q3 tests that rather than assuming it**.

The `ai_*` columns are a casualty I mind: they are the AI-referral signal I looked at in ML-02. They
are excluded on availability grounds, not because they are uninteresting.

In [82]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# The contract as constants, so the code below can never quietly disagree with the words above.
WINDOW_START, WINDOW_END = "2026-03-01", "2026-03-31"
DECISION_MOMENT = "2026-04-01"
SEALED_MONTHS   = ["2026-06"]

# Eligibility for the lane slice (stated here, applied and counted in §3.4).
MIN_IMPRESSIONS = 500    # one click moves CTR by <= 0.2pp at this volume -> the gap is readable
MIN_ACTIVE_DAYS = 5      # volatility and momentum need several days to mean anything
MIN_POSITION    = 1.0    # 1-based floor, applied AFTER the +1 correction in §3.4

CONTRACT = {
    "lane": "Lane 4 - CTR / engagement opportunity scoring",
    "unit_of_analysis": "one page-month: (client_hash_id, content_hash_id) over month=2026-03",
    "source_grain": "report_date x client_hash_id x content_hash_id (page-day)",
    "tables_used": ["fact_content_daily_performance (month=2026-03)", "dim_clients (context only)"],
    "observation_window": f"{WINDOW_START} .. {WINDOW_END}",
    "decision_moment": DECISION_MOMENT,
    "target": "ctr_gap_pp -- AUTHORED PROXY, leave-one-out volume-weighted tier baseline",
    "queue_ordering": "expected_missed_clicks = impressions * ctr_gap_pp / 100",
    "deliberate_exclusion": "all 14 GA4-side columns (three-valued availability flag)",
    "sealed": SEALED_MONTHS,
}
print("DATA CONTRACT (ML-04)\n" + "-" * 72)
for k, v in CONTRACT.items(): print(f"{k:22}: {v}")
print("-" * 72)
print(f"eligibility: impressions >= {MIN_IMPRESSIONS} | active days >= {MIN_ACTIVE_DAYS} | "
      f"avg position >= {MIN_POSITION}")

DATA CONTRACT (ML-04)
------------------------------------------------------------------------
lane                  : Lane 4 - CTR / engagement opportunity scoring
unit_of_analysis      : one page-month: (client_hash_id, content_hash_id) over month=2026-03
source_grain          : report_date x client_hash_id x content_hash_id (page-day)
tables_used           : ['fact_content_daily_performance (month=2026-03)', 'dim_clients (context only)']
observation_window    : 2026-03-01 .. 2026-03-31
decision_moment       : 2026-04-01
target                : ctr_gap_pp -- AUTHORED PROXY, leave-one-out volume-weighted tier baseline
queue_ordering        : expected_missed_clicks = impressions * ctr_gap_pp / 100
deliberate_exclusion  : all 14 GA4-side columns (three-valued availability flag)
sealed                : ['2026-06']
------------------------------------------------------------------------
eligibility: impressions >= 500 | active days >= 5 | avg position >= 1.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

The rule that decides the bucket is the decision moment: a column is a feature only if an editor
could have known it on the morning of 2026-04-01, **and** only if knowing it does not hand back the
answer.

| Bucket | Columns |
|---|---|
| **Feature** (5) | `log_impressions_31d`, `days_with_impressions_31d`, `position_volatility_31d`, `top_day_impression_share`, `momentum_log14v14` |
| **Label / proxy + its ingredients** | `ctr_gap_pp` · and therefore `clicks_31d`, `ctr_pp`, `peer_pp`, tier click/impression sums |
| **Context** (group / join / split / gate — never learned from) | `client_hash_id`, `content_hash_id`, `report_date`, `month`, `impressions_31d`, `avg_position_31d`, `position_tier`, `gsc_data_available`, `ga4_data_available`, `client_has_gsc`, `client_has_ga4`, `gsc_sum_position`, `dim_clients.gsc_data_start` |
| **Excluded** (why below) | all 14 GA4-side columns · `fact_content_query_90d.*` · `dim_content.*` (this week) · `gsc_sum_position` as a feature · FlyRank product flags |

- **The GA4-side columns** — three-valued availability; if it tracks the client, a GA4 feature
  encodes client identity. Excluded until Q3 measures the hole and I can use explicit `has_` flags
  instead of a fill. The `ai_*` family goes with them.
- **`fact_content_query_90d`** — a fixed 90-day window overlapping my future outcome window; its
  `impressions_90d` / `*_last30` columns would contain the label period. Only `*_prev30`-style
  columns could ever be safe, and only after the windows are drawn.
- **`dim_content`** — not excluded on principle (word count, intent, competition are real page
  features) but a join is an untested claim about key integrity, and a bad join does not error, it
  silently returns nothing. It joins in ML-05, after a coverage check.
- **`gsc_sum_position` as a feature** — it is the *numerator* of average position, not an independent
  signal. It earns its place computing position correctly (§3.4) and nowhere else.
- **FlyRank product flags** (`health_score`, `needs_ctr_fix`, …) — already dropped from the release,
  and rightly: they are FlyRank's *answers*, and predicting an answer from the inputs that built it
  is circular success.

**`avg_position_31d` is the interesting one, and it sits in Context on purpose.** Position is
knowable at the decision moment, so this is not temporal leakage. But position *defines the peer
group* whose baseline I subtract — as a feature it would let the model partly reconstruct `peer_pp`,
i.e. half its own target. A structural leak rather than a temporal one. It stays a grouping key, and
ML-05 tests whether it can be promoted.

In [83]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# A contract that checks itself: buckets declared, then tested for overlap.
FEATURES = {
    "log_impressions_31d":       "log1p of total GSC impressions in the window",
    "days_with_impressions_31d": "days in the window with >= 1 impression",
    "position_volatility_31d":   "stddev of daily avg position across active days",
    "top_day_impression_share":  "busiest day's impressions / window impressions",
    "momentum_log14v14":         "log ratio of last-14d vs prior-14d impressions, inside the window",
}
LABEL_AND_INGREDIENTS = {"ctr_gap_pp", "ctr_pp", "peer_pp", "clicks_31d", "tier_clicks",
                         "tier_impressions", "expected_missed_clicks"}
CONTEXT = {"client_hash_id", "content_hash_id", "report_date", "month", "impressions_31d",
           "avg_position_31d", "position_tier", "gsc_sum_position", "gsc_data_start",
           COL["gsc_flag"], COL["ga4_flag"], "client_has_gsc", "client_has_ga4"}
EXCLUDED = {
    "the 14 GA4-side columns (ga4_*, sessions_*, scroll_events, ai_*)":
        "three-valued availability flag -> risks encoding client identity; Q3 measures it",
    "fact_content_query_90d.impressions_90d / *_last30":
        "its fixed 90d window overlaps my future outcome window -> contains the label period",
    "dim_content.* (this week only)":
        "a join is an untested key claim; bad joins fail silently, so ML-05 tests coverage first",
    "gsc_sum_position as a feature":
        "numerator of avg position, not an independent signal -- computes context, never learned from",
    "health_score / needs_ctr_fix / is_quick_win":
        "product ANSWERS, already dropped; predicting them from their own inputs is circular",
}

assert set(FEATURES).isdisjoint(LABEL_AND_INGREDIENTS), "a label ingredient leaked into FEATURES"
assert set(FEATURES).isdisjoint(CONTEXT),               "a context column leaked into FEATURES"

print(f"{len(FEATURES)} features / {len(LABEL_AND_INGREDIENTS)} label-side / {len(CONTEXT)} context / "
      f"{len(EXCLUDED)} exclusion rules")
print("bucket disjointness: PASS\n")
for k, why in EXCLUDED.items(): print(f"EXCLUDED  {k}\n          why: {why}")

5 features / 7 label-side / 13 context / 5 exclusion rules
bucket disjointness: PASS

EXCLUDED  the 14 GA4-side columns (ga4_*, sessions_*, scroll_events, ai_*)
          why: three-valued availability flag -> risks encoding client identity; Q3 measures it
EXCLUDED  fact_content_query_90d.impressions_90d / *_last30
          why: its fixed 90d window overlaps my future outcome window -> contains the label period
EXCLUDED  dim_content.* (this week only)
          why: a join is an untested key claim; bad joins fail silently, so ML-05 tests coverage first
EXCLUDED  gsc_sum_position as a feature
          why: numerator of avg position, not an independent signal -- computes context, never learned from
EXCLUDED  health_score / needs_ctr_fix / is_quick_win
          why: product ANSWERS, already dropped; predicting them from their own inputs is circular


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

| # | The claim from §1 | The query that could disprove it |
|---|---|---|
| **Q1** | the source grain is one row per page per day per client | group by those three columns, look for a group with more than one row |
| **Q2** | my slice is one calendar month, 2026-03-01 → 2026-03-31 | row count, distinct pages/clients, `MIN`/`MAX(report_date)`, and the `month` value stamped in the rows |
| **Q3** | availability is three-valued, and GA4 availability tracks the client | count TRUE / FALSE / NULL, show what `= FALSE` drops, and compare the client flag against the row flag |

Each cell prints a verdict sentence with its number in it, so claim and evidence stay attached.

### Q1 — Grain: is one row really one page × one day × one client?

If the grain holds, grouping by those three columns can never produce a group with more than one
row. Zero rows back = the grain holds.

In [84]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- Q1: grain probe --------------------------------------------------------
q1 = con.sql(f"""
    SELECT {COL['date']}, {COL['client']}, {COL['content']}, COUNT(*) AS n
    FROM {T['fact_month']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"groups with more than one row (showing up to 5): {len(q1)}")
print("VERDICT: grain holds -- one row = one page x one day x one client, so my page-month row is a "
      "roll-up of up to 31 of these."
      if len(q1) == 0 else
      "VERDICT: grain does NOT hold -- the sentence in section 1 must be rewritten.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

groups with more than one row (showing up to 5): 0
VERDICT: grain holds -- one row = one page x one day x one client, so my page-month row is a roll-up of up to 31 of these.


### Q2 — Counts and window: how big is the slice, and does it really cover one month?

A partition folder is a *promise* about dates, not a proof. This checks the dates themselves — and
the `month` value stamped inside the rows, so the folder name and the data have to agree.

In [85]:
# --- Q2: row count, distinct pages/clients, date span -----------------------
month_sel = (",\n           COUNT(DISTINCT month) AS month_values,\n           ANY_VALUE(month) AS month_stamp"
             if HAS_MONTH_COL else "")
q2 = con.sql(f"""
    SELECT COUNT(*)                            AS page_day_rows,
           COUNT(DISTINCT {COL['content']})    AS distinct_pages,
           COUNT(DISTINCT {COL['client']})     AS distinct_clients,
           MIN({COL['date']})                  AS first_date,
           MAX({COL['date']})                  AS last_date,
           COUNT(DISTINCT {COL['date']})       AS distinct_days{month_sel}
    FROM {T['fact_month']}
""").df().iloc[0]

span_ok = str(q2.first_date)[:10] == WINDOW_START and str(q2.last_date)[:10] == WINDOW_END
print(f"page-day rows   : {q2.page_day_rows:,}")
print(f"distinct pages  : {q2.distinct_pages:,}   distinct clients: {q2.distinct_clients:,}")
print(f"date span       : {str(q2.first_date)[:10]} -> {str(q2.last_date)[:10]} "
      f"({q2.distinct_days} days) | avg {q2.page_day_rows / q2.distinct_pages:.1f} page-days per page")
if HAS_MONTH_COL:
    print(f"month stamped in rows: {q2.month_stamp} ({q2.month_values} distinct) -> folder name and data "
          f"{'AGREE' if (q2.month_values == 1 and str(q2.month_stamp) == MONTH) else 'DISAGREE'}")
print(f"\nVERDICT: the dates {'match' if span_ok else 'DO NOT match'} the window claimed in section 1. "
      f"Rolled up to page-month grain that is at most {q2.distinct_pages:,} rows, before eligibility.")

page-day rows   : 9,841,378
distinct pages  : 331,437   distinct clients: 55
date span       : 2026-03-01 -> 2026-03-31 (31 days) | avg 29.7 page-days per page
month stamped in rows: 2026-03 (1 distinct) -> folder name and data AGREE

VERDICT: the dates match the window claimed in section 1. Rolled up to page-month grain that is at most 331,437 rows, before eligibility.


### Q3 — Availability: the two kinds of nothing

A `0` in an impressions column means *we looked and there was nothing*. A `NULL` means *we could
not look*. The flags that tell them apart are three-valued, so `= TRUE` is not a filter — it is a
bug that returns fewer rows than you think.

This also tests §1's assumption rather than trusting it: if `client_has_ga4` and `ga4_data_available`
almost always agree, GA4 availability is a **client** property and the exclusion stands as written.
If they disagree often, the exclusion still holds but my stated reason was too narrow.

In [86]:
# --- Q3: availability funnel, filtered with IS TRUE -------------------------
client_flag_sel = ""
if HAS_CLIENT_FLAGS:
    client_flag_sel = (",\n           COUNT(*) FILTER (WHERE client_has_ga4 IS TRUE) AS ga4_client_true"
                       ",\n           COUNT(*) FILTER (WHERE client_has_ga4 IS TRUE AND "
                       "ga4_data_available IS NOT TRUE) AS ga4_client_on_row_off"
                       ",\n           COUNT(*) FILTER (WHERE client_has_gsc IS TRUE AND "
                       "gsc_data_available IS NOT TRUE) AS gsc_client_on_row_off")

q3 = con.sql(f"""
    SELECT COUNT(*)                                                   AS all_rows,
           COUNT(*) FILTER (WHERE {COL['gsc_flag']} IS TRUE)          AS gsc_true,
           COUNT(*) FILTER (WHERE {COL['gsc_flag']} IS FALSE)         AS gsc_false,
           COUNT(*) FILTER (WHERE {COL['gsc_flag']} IS NULL)          AS gsc_null,
           COUNT(*) FILTER (WHERE {COL['ga4_flag']} IS TRUE)          AS ga4_true,
           COUNT(*) FILTER (WHERE {COL['ga4_flag']} IS FALSE)         AS ga4_false,
           COUNT(*) FILTER (WHERE {COL['ga4_flag']} IS NULL)          AS ga4_null,
           COUNT(*) FILTER (WHERE {COL['gsc_flag']} IS TRUE
                              AND {COL['imp']} > 0)                   AS gsc_true_with_imp{client_flag_sel}
    FROM {T['fact_month']}
""").df().iloc[0]

n = q3.all_rows
p = lambda x: f"{x:>12,} ({x / n:6.2%})"
print(f"rows in month={MONTH}: {n:,}\n")
print(f"gsc_data_available  TRUE {p(q3.gsc_true)} | FALSE {p(q3.gsc_false)} | NULL {p(q3.gsc_null)}")
print(f"ga4_data_available  TRUE {p(q3.ga4_true)} | FALSE {p(q3.ga4_false)} | NULL {p(q3.ga4_null)}")
print()
print("The trap, in numbers:")
print(f"  ga4_data_available = FALSE     catches {q3.ga4_false:,} rows")
print(f"  ga4_data_available IS NOT TRUE catches {q3.ga4_false + q3.ga4_null:,} rows")
print(f"  a '= FALSE' filter silently drops {q3.ga4_null:,} NULL rows -- {q3.ga4_null / n:.1%} of the month")
print()
# What does the GSC flag actually MEAN? Compare it with "had impressions".
print(f"GSC flag IS TRUE                    : {q3.gsc_true:,}")
print(f"GSC flag IS TRUE and impressions > 0: {q3.gsc_true_with_imp:,}")
if q3.gsc_true == q3.gsc_true_with_imp:
    print("  -> identical. OBSERVED: gsc_data_available is not 'this client has Search Console', it is "
          "'this page-day actually had search data'. That makes it a ROW-level flag, and it is doing "
          "the volume filtering I assumed my own rules would do (see the funnel in 3.4).")
if HAS_CLIENT_FLAGS:
    client_on = q3.ga4_client_true
    off = q3.ga4_client_on_row_off / client_on
    print(f"\nrows where client_has_ga4 IS TRUE       : {client_on:,}")
    print(f"  of which ga4_data_available IS TRUE   : {q3.ga4_true:,}")
    print(f"  of which IS NOT TRUE                  : {q3.ga4_client_on_row_off:,} ({off:.1%})")
    print(f"rows where client_has_ga4 is NOT TRUE   : {n - client_on:,}")
    print(f"  of which ga4_data_available IS NULL   : {q3.ga4_null:,}")
    if (n - client_on) == q3.ga4_null:
        print("  -> the two flags PARTITION the month exactly, with no remainder: NULL means 'this "
              "client is not instrumented -- we could not look'; FALSE means 'the client IS "
              "instrumented but this page-day had nothing'. They never contradict each other.")
    print(f"  -> OBSERVED: within instrumented clients, {off:.1%} of page-days still carry no GA4 "
          f"data, so availability is overwhelmingly a PAGE-DAY property, not a client one. Section "
          f"1's exclusion stands, and for a stronger reason than I wrote. Note also that FALSE holds "
          f"two meanings at once -- before ga4_data_start (could not look) and after it with no "
          f"traffic (looked, nothing there) -- one flag value, both kinds of nothing.")
    print(f"\nGSC twin: client_has_gsc IS TRUE but gsc_data_available IS NOT TRUE: "
          f"{q3.gsc_client_on_row_off:,}")
print("\nVERDICT: availability is three-valued, so every filter here uses IS TRUE / IS NOT TRUE, and "
      "the section 1 exclusion is now a measured decision rather than a hunch.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows in month=2026-03: 9,841,378

gsc_data_available  TRUE    3,611,061 (36.69%) | FALSE    6,230,317 (63.31%) | NULL            0 ( 0.00%)
ga4_data_available  TRUE      413,966 ( 4.21%) | FALSE    6,408,671 (65.12%) | NULL    3,018,741 (30.67%)

The trap, in numbers:
  ga4_data_available = FALSE     catches 6,408,671 rows
  ga4_data_available IS NOT TRUE catches 9,427,412 rows
  a '= FALSE' filter silently drops 3,018,741 NULL rows -- 30.7% of the month

GSC flag IS TRUE                    : 3,611,061
GSC flag IS TRUE and impressions > 0: 3,611,061
  -> identical. OBSERVED: gsc_data_available is not 'this client has Search Console', it is 'this page-day actually had search data'. That makes it a ROW-level flag, and it is doing the volume filtering I assumed my own rules would do (see the funnel in 3.4).

rows where client_has_ga4 IS TRUE       : 6,822,637
  of which ga4_data_available IS TRUE   : 413,966
  of which IS NOT TRUE                  : 6,408,671 (93.9%)
rows where client_has

### 3.4 A sanity check the contract nearly skipped

Before building anything on position, one probe — because `avg_position = 0` meaning "no data" was
already a documented trap, and a column that lies in one way may lie in another.

The probe checks two things: how many positions fall below 1, and whether `gsc_avg_position` equals
`gsc_sum_position / gsc_impressions`. The second question is the load-bearing one — Search Console's
bulk export stores `sum_position` **zero-based** (0 = the top result) and documents 1-based average
position as `SUM(sum_position)/SUM(impressions) + 1`. So if the column matches sum ÷ impressions
exactly, it is carrying Google's zero-based convention and every position in this notebook needs a
`+1`. That also makes sub-1 values legitimate rather than corrupt: they are the *best*-ranked pages.

In [87]:
# --- Column sanity: is gsc_avg_position what its name promises? -------------
# (A data-quality probe, not one of the three contract queries above.)
sanity = con.sql(f"""
    SELECT COUNT(*)                                                     AS rows_with_pos,
           COUNT(*) FILTER (WHERE {COL['pos']} < 1)                     AS pos_below_1,
           MIN({COL['pos']})                                            AS min_pos,
           quantile_cont({COL['pos']}, 0.5)                             AS median_pos,
           COUNT(*) FILTER (WHERE ABS({COL['pos']} - gsc_sum_position /
                            NULLIF({COL['imp']}, 0)) > 0.01)            AS avg_vs_sum_disagrees
    FROM {T['fact_month']}
    WHERE {COL['gsc_flag']} IS TRUE AND {COL['pos']} IS NOT NULL AND {COL['imp']} > 0
""").df().iloc[0] if HAS_SUM_POSITION else None

if sanity is not None:
    print(f"rows with a position           : {sanity.rows_with_pos:,}")
    print(f"positions below 1 (impossible) : {sanity.pos_below_1:,} "
          f"({sanity.pos_below_1 / sanity.rows_with_pos:.2%})")
    print(f"min / median position          : {sanity.min_pos:.3f} / {sanity.median_pos:.2f}")
    print(f"rows where gsc_avg_position != gsc_sum_position/gsc_impressions: "
          f"{sanity.avg_vs_sum_disagrees:,}")
    print()
    if sanity.avg_vs_sum_disagrees == 0:
        print("OBSERVED: gsc_avg_position equals gsc_sum_position / gsc_impressions on EVERY row -- "
              "which is precisely the quantity Search Console's bulk export documents as ZERO-BASED. "
              "So the sub-1 values are not corrupt: a page reading 0.06 sits at true position 1.06, "
              "and those are the best-ranked pages in the slice. Every position below is therefore "
              "computed with the +1 correction applied, and MIN_POSITION becomes a guard that should "
              "now remove nothing.")
        print(f"\nFollow-through: a `position > 0` guard would have been correct under the starter "
              f"CSV's convention ('0 means no data') and is WRONG here -- zero is rank 1. Missing "
              f"position is NULL in this table, which the filter already handles, so the guard is "
              f"dropped and only `IS NOT NULL` remains.")
    else:
        print("OBSERVED: gsc_avg_position is NOT sum/impressions -- it is derived some other way, so "
              "gsc_sum_position is the safer numerator. Raise this in the async channel before "
              "trusting any tier boundary.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows with a position           : 3,611,061.0
positions below 1 (impossible) : 264,737.0 (7.33%)
min / median position          : 0.000 / 7.50
rows where gsc_avg_position != gsc_sum_position/gsc_impressions: 0.0

OBSERVED: gsc_avg_position equals gsc_sum_position / gsc_impressions on EVERY row -- which is precisely the quantity Search Console's bulk export documents as ZERO-BASED. So the sub-1 values are not corrupt: a page reading 0.06 sits at true position 1.06, and those are the best-ranked pages in the slice. Every position below is therefore computed with the +1 correction applied, and MIN_POSITION becomes a guard that should now remove nothing.

Follow-through: a `position > 0` guard would have been correct under the starter CSV's convention ('0 means no data') and is WRONG here -- zero is rank 1. Missing position is NULL in this table, which the filter already handles, so the guard is dropped and only `IS NOT NULL` remains.


### The lane slice and the five features

One query aggregates page-days into page-months for **every** page with GSC data — the eligibility
rules are then applied in pandas so the funnel is visible instead of hidden inside a `WHERE`.
Everything is bounded by `report_date <= 2026-03-31`, so everything survives the decision-moment test.

In [88]:
# --- page-days -> page-months (no eligibility filter yet) -------------------
# Momentum splits the window into two equal halves: Mar 18-31 vs Mar 04-17 (Mar 01-03 unused so the
# halves are the same length). Position numerator: gsc_sum_position when available.
LAST14_FROM, PREV14_FROM = "2026-03-18", "2026-03-04"
pos_num = "SUM(gsc_sum_position)" if HAS_SUM_POSITION else f"SUM({COL['pos']} * {COL['imp']})"
POS_OK  = f"{COL['imp']} > 0 AND {COL['pos']} IS NOT NULL"

pages = con.sql(f"""
    SELECT {COL['client']}  AS client_hash_id,
           {COL['content']} AS content_hash_id,
           SUM({COL['imp']})                                                AS impressions_31d,
           SUM({COL['clicks']})                                             AS clicks_31d,
           COUNT(DISTINCT CASE WHEN {COL['imp']} > 0 THEN {COL['date']} END) AS days_with_impressions_31d,
           MAX({COL['imp']})                                                AS top_day_impressions,
           {pos_num}          FILTER (WHERE {POS_OK})                       AS pos_num,
           SUM({COL['imp']})  FILTER (WHERE {POS_OK})                       AS pos_den,
           STDDEV_SAMP({COL['pos']}) FILTER (WHERE {POS_OK})                AS position_volatility_31d,
           SUM({COL['imp']}) FILTER (WHERE {COL['date']} >= DATE '{LAST14_FROM}') AS imp_last14,
           SUM({COL['imp']}) FILTER (WHERE {COL['date']} >= DATE '{PREV14_FROM}'
                                       AND {COL['date']} <  DATE '{LAST14_FROM}') AS imp_prev14
    FROM {T['fact_month']}
    WHERE {COL['gsc_flag']} IS TRUE          -- IS TRUE, never = TRUE
    GROUP BY 1, 2
""").df()

# +1: Search Console's sum_position is zero-based (see the probe in 3.4), so 1-based = sum/imp + 1
pages["avg_position_31d"] = pages["pos_num"] / pages["pos_den"].replace(0, np.nan) + 1
print(f"pages with GSC data in the window: {len(pages):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages with GSC data in the window: 176,738


In [89]:
# --- The eligibility funnel, in the open ------------------------------------
k1 = pages["impressions_31d"] >= MIN_IMPRESSIONS
k2 = k1 & (pages["days_with_impressions_31d"] >= MIN_ACTIVE_DAYS)
k3 = k2 & pages["avg_position_31d"].notna()
k4 = k3 & (pages["avg_position_31d"] >= MIN_POSITION)
lane = pages[k4].copy()

print("Eligibility funnel (pages)")
print(f"  GSC availability IS TRUE      : {len(pages):>8,}")
print(f"  >= {MIN_IMPRESSIONS} impressions           : {k1.sum():>8,}  (-{len(pages)-k1.sum():,})")
print(f"  >= {MIN_ACTIVE_DAYS} active days              : {k2.sum():>8,}  (-{k1.sum()-k2.sum():,})")
print(f"  has a usable position         : {k3.sum():>8,}  (-{k2.sum()-k3.sum():,})")
print(f"  position >= {MIN_POSITION}              : {k4.sum():>8,}  (-{k3.sum()-k4.sum():,})   <- lane slice")
print()
print(f"OBSERVED: the volume floor does nearly all the work; the later rules remove very few pages. "
      f"That is not because they are strict but because gsc_data_available already restricts rows to "
      f"page-days with search data (Q3). I keep them as explicit guards -- a rule that is a no-op "
      f"today is not a no-op after a schema change -- but I am not claiming they filter anything.")
print(f"\nScope, stated: my lane speaks only about the {len(lane):,} pages already visible enough for a "
      f"CTR gap to be readable, and says nothing about the {len(pages)-len(lane):,} below the floor.")
print(f"\nThe position rule now removes {int(k3.sum()-k4.sum())} pages -- correctly. Before the +1 "
      f"correction it removed 452, and every one of them was a TOP-ranked page. A guard written to "
      f"catch impossible data was quietly deleting my best rows; it took the 3.4 probe to see it.")

Eligibility funnel (pages)
  GSC availability IS TRUE      :  176,738
  >= 500 impressions           :   61,924  (-114,814)
  >= 5 active days              :   61,881  (-43)
  has a usable position         :   61,881  (-0)
  position >= 1.0              :   61,881  (-0)   <- lane slice

OBSERVED: the volume floor does nearly all the work; the later rules remove very few pages. That is not because they are strict but because gsc_data_available already restricts rows to page-days with search data (Q3). I keep them as explicit guards -- a rule that is a no-op today is not a no-op after a schema change -- but I am not claiming they filter anything.

Scope, stated: my lane speaks only about the 61,881 pages already visible enough for a CTR gap to be readable, and says nothing about the 114,857 below the floor.

The position rule now removes 0 pages -- correctly. Before the +1 correction it removed 452, and every one of them was a TOP-ranked page. A guard written to catch impossible data was

In [90]:
# --- Five features + the proxy ---------------------------------------------
def position_tier(p):                      # documented thresholds: <=3 / <=10 / <=20 / <=50 / >50
    return ("top_3" if p <= 3 else "page_1" if p <= 10 else "striking" if p <= 20
            else "page_3_5" if p <= 50 else "deep")

lane["position_tier"] = lane["avg_position_31d"].apply(position_tier)      # CONTEXT (grouping key)

lane["log_impressions_31d"]      = np.log1p(lane["impressions_31d"])
lane["top_day_impression_share"] = lane["top_day_impressions"] / lane["impressions_31d"]
# fillna(0) is legitimate here and only here: the GSC flag IS TRUE and the days exist in the window,
# so an empty half means "we looked and there was nothing" -- the 0 kind of nothing.
lane["momentum_log14v14"] = np.log((lane["imp_last14"].fillna(0) + 1) /
                                   (lane["imp_prev14"].fillna(0) + 1))

tier_clicks = lane.groupby("position_tier")["clicks_31d"].transform("sum")
tier_imps   = lane.groupby("position_tier")["impressions_31d"].transform("sum")
lane["ctr_pp"]  = 100 * lane["clicks_31d"] / lane["impressions_31d"]
lane["peer_pp"] = (100 * (tier_clicks - lane["clicks_31d"]) /
                   (tier_imps - lane["impressions_31d"])).replace([np.inf, -np.inf], np.nan)
lane["ctr_gap_pp"] = lane["peer_pp"] - lane["ctr_pp"]                      # THE PROXY
lane["expected_missed_clicks"] = lane["impressions_31d"] * lane["ctr_gap_pp"] / 100

before = len(lane)
lane = lane.dropna(subset=list(FEATURES) + ["ctr_gap_pp"]).copy()
print(f"dropped for a missing feature or proxy value: {before - len(lane):,} (never filled)")
print(f"modelling frame: {len(lane):,} pages, {lane.client_hash_id.nunique()} clients\n")

g = lane.groupby("position_tier")
tiers = g.agg(pages=("ctr_gap_pp", "size"), _clicks=("clicks_31d", "sum"),
              _imps=("impressions_31d", "sum"), median_gap_pp=("ctr_gap_pp", "median"))
tiers["pooled_ctr_pp"]        = 100 * tiers.pop("_clicks") / tiers.pop("_imps")
tiers["share_underperforming"] = g["ctr_gap_pp"].apply(lambda s: (s > 0).mean())
tiers = tiers[["pages", "pooled_ctr_pp", "median_gap_pp", "share_underperforming"]].round(3)
display(tiers)

DOC_TOP3_CTR = 2.78   # docs: at warehouse scale positions 1-3 run ~2.78% CTR
pooled_all = 100 * lane.clicks_31d.sum() / lane.impressions_31d.sum()
order = [t for t in ["top_3", "page_1", "striking", "page_3_5", "deep"] if t in tiers.index]
seq = [tiers.loc[t, "pooled_ctr_pp"] for t in order]
monotonic = all(a >= b for a, b in zip(seq, seq[1:]))

print(f"pooled CTR across the whole lane : {pooled_all:.3f}%")
print(f"CTR falls monotonically top_3 -> deep: {monotonic}")
print()
print(f"UNRESOLVED, and reframed by the numbers above: my top_3 pooled CTR is "
      f"{tiers.loc['top_3','pooled_ctr_pp']:.3f}% against ~{DOC_TOP3_CTR}% in the data dictionary, "
      f"but the whole lane pools to {pooled_all:.3f}% -- so this is a LEVEL problem across every "
      f"tier, not a top_3 artifact, and not an off-by-one (positions are 1-based here). "
      + ("Worse, CTR does not fall monotonically: top_3 sits BELOW page_1, which position alone "
         "cannot explain. " if not monotonic else "")
      + "Candidate explanations I cannot separate with this table: impressions counted on a "
      "different basis from clicks, monthly page-level averaging mixing many queries per page, or "
      "a tier dominated by a few atypical clients. The queue ranks WITHIN tier so it still works, "
      "but no absolute tier CTR from this notebook should be quoted as a fact about Google. Async "
      "channel.")

dropped for a missing feature or proxy value: 0 (never filled)
modelling frame: 61,881 pages, 36 clients



,pages,pooled_ctr_pp,median_gap_pp,share_underperforming
position_tier,,,,
deep,749,0.036,0.036,0.752
page_1,35203,0.340,0.115,0.651
page_3_5,11152,0.145,0.059,0.650
striking,11916,0.330,0.161,0.713
top_3,2861,0.317,0.133,0.687


pooled CTR across the whole lane : 0.295%
CTR falls monotonically top_3 -> deep: False

UNRESOLVED, and reframed by the numbers above: my top_3 pooled CTR is 0.317% against ~2.78% in the data dictionary, but the whole lane pools to 0.295% -- so this is a LEVEL problem across every tier, not a top_3 artifact, and not an off-by-one (positions are 1-based here). Worse, CTR does not fall monotonically: top_3 sits BELOW page_1, which position alone cannot explain. Candidate explanations I cannot separate with this table: impressions counted on a different basis from clicks, monthly page-level averaging mixing many queries per page, or a tier dominated by a few atypical clients. The queue ranks WITHIN tier so it still works, but no absolute tier CTR from this notebook should be quoted as a fact about Google. Async channel.


In [91]:
# --- Did my two ML-03 limitations survive contact with real data? -----------
# (a) ML-03: zero-CTR pages all TIE at the maximum score regardless of volume.
v1_ties = (lane["ctr_gap_pp"] == lane["ctr_gap_pp"].max()).sum()
zero_clicks = lane[lane["clicks_31d"] == 0]
print(f"pages with zero clicks: {len(zero_clicks):,} ({len(zero_clicks)/len(lane):.1%}), "
      f"impressions {zero_clicks.impressions_31d.min():,.0f} .. {zero_clicks.impressions_31d.max():,.0f}")
print(f"pages tied at the maximum rate gap: {v1_ties:,}")
print("  -> CORRECTION to ML-03: the tie problem was an artifact of the starter CSV, whose ctr column "
      "is rounded to 2 decimals and therefore manufactures exact ties. The warehouse computes CTR "
      "from raw counts, so exact ties essentially vanish.\n")

# The real problem is not ties, it is composition -- so measure that instead.
top_rate = lane.nlargest(50, "ctr_gap_pp")
top_vol  = lane.nlargest(50, "expected_missed_clicks")
comp = pd.DataFrame({
    "ranked by rate gap":      [top_rate.impressions_31d.median(),
                                (top_rate.clicks_31d == 0).mean(),
                                top_rate.expected_missed_clicks.sum()],
    "ranked by missed clicks": [top_vol.impressions_31d.median(),
                                (top_vol.clicks_31d == 0).mean(),
                                top_vol.expected_missed_clicks.sum()],
}, index=["median impressions", "share with zero clicks", "total missed clicks in top 50"]).round(2)
display(comp)
overlap = len(set(top_rate.index) & set(top_vol.index))
ratio   = top_vol.impressions_31d.median() / max(top_rate.impressions_31d.median(), 1)
print(f"OBSERVED: the two top-50 lists share {overlap} of 50 pages. The volume-weighted queue's median "
      f"page carries {ratio:.1f}x the impressions of the rate-gap queue's, and it collects "
      f"{top_vol.expected_missed_clicks.sum() / max(top_rate.expected_missed_clicks.sum(), 1):.1f}x the "
      f"missed clicks. That is the ML-03 limitation closed -- as composition, not as ties: the rate "
      f"gap alone cannot tell a big opportunity from a tiny one.\n")

# (b) ML-03: the tier baseline included the page being scored (circularity).
naive = lane.groupby("position_tier")["ctr_pp"].transform("mean")   # ML-03 style: self-included mean of rates
shift = (naive - lane["peer_pp"]).abs()
print(f"(b) |ML-03 baseline - leave-one-out baseline|: median {shift.median():.3f}pp, "
      f"max {shift.max():.3f}pp")
print("  -> OBSERVED, directional: the circularity I flagged in ML-03 was real but small at this "
      "scale. Worth fixing, not worth having worried about -- and now measured rather than assumed.")

pages with zero clicks: 10,783 (17.4%), impressions 500 .. 44,707
pages tied at the maximum rate gap: 1
  -> CORRECTION to ML-03: the tie problem was an artifact of the starter CSV, whose ctr column is rounded to 2 decimals and therefore manufactures exact ties. The warehouse computes CTR from raw counts, so exact ties essentially vanish.



,ranked by rate gap,ranked by missed clicks
median impressions,7936.50,80152.50
share with zero clicks,1.00,0.00
total missed clicks in top 50,1680.86,11864.25


OBSERVED: the two top-50 lists share 0 of 50 pages. The volume-weighted queue's median page carries 10.1x the impressions of the rate-gap queue's, and it collects 7.1x the missed clicks. That is the ML-03 limitation closed -- as composition, not as ties: the rate gap alone cannot tell a big opportunity from a tiny one.

(b) |ML-03 baseline - leave-one-out baseline|: median 0.010pp, max 0.057pp
  -> OBSERVED, directional: the circularity I flagged in ML-03 was real but small at this scale. Worth fixing, not worth having worried about -- and now measured rather than assumed.


### 3.5 The five features — "knowable at the decision moment because…"

| # | Feature | Knowable at the decision moment because… |
|---|---|---|
| 1 | `log_impressions_31d` | it sums impressions with `report_date <= 2026-03-31`; the last day it sees is the day before the queue is built. `log1p` for the heavy tail — a transform adds no information the column lacked. |
| 2 | `days_with_impressions_31d` | it counts days *inside* the window that had at least one impression — a coverage measure of the past. It was the strongest feature in the starter pipeline, so it is signal, not filler. |
| 3 | `position_volatility_31d` | the stddev of daily positions already recorded in the window. Days with no impressions are excluded, and missing position is `NULL` — not `0`, which under this table's zero-based convention is rank 1, the best result there is. |
| 4 | `top_day_impression_share` | the busiest day's share of window impressions, from the same 31 days. It separates one spike from steady demand — two situations whose CTR should not be read the same way. |
| 5 | `momentum_log14v14` | both halves (Mar 18–31, Mar 04–17) sit **inside** the window. Momentum measured in the past, not a peek at April. `log((a+1)/(b+1))` stays defined when a half is zero without inventing a value. |

**What is not here, and why that is the point.** `clicks_31d` is equally knowable on 2026-04-01 — and
disqualified anyway, because the proxy is arithmetic on it. Knowability is necessary, not sufficient.
§3.6 shows what happens when I forget that.

In [92]:
print(f"decision moment {DECISION_MOMENT} -- every feature computed from report_date <= {WINDOW_END}\n")
for i, (name, what) in enumerate(FEATURES.items(), 1): print(f"{i}. {name:28} {what}")
print()
print(lane[list(FEATURES)].describe().round(3))

decision moment 2026-04-01 -- every feature computed from report_date <= 2026-03-31

1. log_impressions_31d          log1p of total GSC impressions in the window
2. days_with_impressions_31d    days in the window with >= 1 impression
3. position_volatility_31d      stddev of daily avg position across active days
4. top_day_impression_share     busiest day's impressions / window impressions
5. momentum_log14v14            log ratio of last-14d vs prior-14d impressions, inside the window

       log_impressions_31d  days_with_impressions_31d  position_volatility_31d  top_day_impression_share  momentum_log14v14
count            61881.000                  61881.000                61881.000                 61881.000          61881.000
mean                 7.710                     29.533                    4.494                     0.083              0.234
std                  1.046                      3.831                    4.014                     0.053              1.268
min         

### 3.6 The trap: one deliberate leak, then removed

Notebook 02 demonstrated leakage on the starter CSV with `trend_pct`. Same lesson, real warehouse
data, and a column that looks far more innocent.

**The leak: `clicks_31d`.** It passes every test a beginner applies — a real observed metric,
knowable at the decision moment, and "how many clicks did this page get" sounds like an obvious page
feature. But the proxy is `ctr_gap_pp = peer_pp − 100 × clicks_31d / impressions_31d`, and
`impressions_31d` is already a feature while `peer_pp` takes only five tier-level values. Hand the
model `clicks_31d` and it stops predicting: it is doing arithmetic it has been given both operands
for.

Scored on **held-out clients** (mirroring the starter pipeline's `client_holdout`), with Spearman as
the headline because this lane is a ranking problem — what matters is the order, not the value.

In [93]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

y, groups = lane["ctr_gap_pp"].values, lane["client_hash_id"].values
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(lane, y, groups))
print(f"grouped split: {len(tr):,} train / {len(te):,} test rows "
      f"({len(te)/len(lane):.0%} of rows) | {lane.iloc[tr].client_hash_id.nunique()} train clients / "
      f"{lane.iloc[te].client_hash_id.nunique()} held-out clients, none shared")

def quick_score(cols, label):
    X = lane[cols].values
    p = RandomForestRegressor(n_estimators=120, random_state=42, n_jobs=-1).fit(X[tr], y[tr]).predict(X[te])
    rho, r2 = spearmanr(p, y[te])[0], r2_score(y[te], p)
    print(f"{label:<34} Spearman {rho:6.3f} | R2 {r2:7.3f}")
    return rho, r2

honest_cols = list(FEATURES)
print("\nheld-out-client scores\n" + "-" * 62)
rho_h, r2_h = quick_score(honest_cols,                  "5 honest features")
rho_l, r2_l = quick_score(honest_cols + ["clicks_31d"], "+ clicks_31d  (DELIBERATE LEAK)")
print("-" * 62)
print(f"the leak buys {rho_l - rho_h:+.3f} Spearman and {r2_l - r2_h:+.3f} R2 -- every point of it "
      f"arithmetic, not signal.")

grouped split: 33,676 train / 28,205 test rows (46% of rows) | 27 train clients / 9 held-out clients, none shared

held-out-client scores
--------------------------------------------------------------
5 honest features                  Spearman  0.167 | R2  -0.165
+ clicks_31d  (DELIBERATE LEAK)    Spearman  0.902 | R2   0.942
--------------------------------------------------------------
the leak buys +0.735 Spearman and +1.107 R2 -- every point of it arithmetic, not signal.


In [94]:
# --- Leak removed; keep the honest number and read it properly --------------
assert "clicks_31d" not in FEATURES and set(FEATURES).isdisjoint(LABEL_AND_INGREDIENTS)
HONEST_BASELINE = {"features": honest_cols, "spearman": round(float(rho_h), 3),
                   "r2": round(float(r2_h), 3), "validation": "GroupShuffleSplit on client_hash_id",
                   "seed": 42}
print(f"leak removed. Carried forward: Spearman {rho_h:.3f} | R2 {r2_h:.3f} on held-out clients, "
      f"5 features, seed 42\n")

if rho_h > 0 and r2_h < 0:
    print("Reading it honestly: positive Spearman with NEGATIVE R2 is a specific, nameable result -- "
          "weak RANK information transfers to unseen clients, while MAGNITUDE does not transfer at "
          "all (the model does worse than predicting the test mean). That is a level shift between "
          "clients: what counts as a normal CTR gap differs by client. It is direct evidence for the "
          "limitation in section 4, and it points at per-client normalisation in ML-05.")
else:
    print("Reading it honestly: this is a floor, not a result -- how far five volume-and-stability "
          "features can order an AUTHORED proxy on clients the model never saw.")

top3_share = (lane.groupby("client_hash_id").size().sort_values(ascending=False) / len(lane)).head(3).sum()
print(f"\nCaveat on the split: GroupShuffleSplit divides CLIENTS, not rows, and the top 3 clients hold "
      f"{top3_share:.0%} of rows -- so a nominal 25% test size actually returned {len(te)/len(lane):.0%} "
      f"of rows. With that concentration a single grouped split is high-variance whichever way it "
      f"lands; ML-05 should report GroupKFold across folds rather than one number.")

leak removed. Carried forward: Spearman 0.167 | R2 -0.165 on held-out clients, 5 features, seed 42

Reading it honestly: positive Spearman with NEGATIVE R2 is a specific, nameable result -- weak RANK information transfers to unseen clients, while MAGNITUDE does not transfer at all (the model does worse than predicting the test mean). That is a level shift between clients: what counts as a normal CTR gap differs by client. It is direct evidence for the limitation in section 4, and it points at per-client normalisation in ML-05.

Caveat on the split: GroupShuffleSplit divides CLIENTS, not rows, and the top 3 clients hold 57% of rows -- so a nominal 25% test size actually returned 46% of rows. With that concentration a single grouped split is high-variance whichever way it lands; ML-05 should report GroupKFold across folds rather than one number.


### 3.7 Is the queue worth an editor's time?

A contract should say what the analysis hands to the human. Before committing to this lane for the
capstone, the honest question is whether the top of the queue is worth opening at all — a gap of
0.1pp on a small page is real and worthless.

In [95]:
for K in (50, 200):
    top = lane.nlargest(K, "expected_missed_clicks")
    print(f"top {K:>3} by expected missed clicks: {top.expected_missed_clicks.sum():>9,.0f} clicks/month "
          f"| median page {top.impressions_31d.median():>8,.0f} impressions, gap {top.ctr_gap_pp.median():.2f}pp "
          f"| {top.client_hash_id.nunique()} clients")
actual = lane["clicks_31d"].sum()
top50 = lane.nlargest(50, "expected_missed_clicks")["expected_missed_clicks"].sum()
print(f"\nfor scale: the lane's pages actually earned {actual:,.0f} clicks in the window, so the top-50 "
      f"queue is worth about {top50 / actual:.1%} of observed monthly clicks -- ESTIMATED under the "
      f"proxy's assumption that a page could reach its tier's peer CTR, which no experiment here "
      f"tests. Directional sizing for whether the lane is worth building, not a promise of recovery.")

top  50 by expected missed clicks:    11,864 clicks/month | median page   80,152 impressions, gap 0.27pp | 9 clients
top 200 by expected missed clicks:    28,365 clicks/month | median page   53,816 impressions, gap 0.25pp | 12 clients

for scale: the lane's pages actually earned 792,668 clicks in the window, so the top-50 queue is worth about 1.5% of observed monthly clicks -- ESTIMATED under the proxy's assumption that a page could reach its tier's peer CTR, which no experiment here tests. Directional sizing for whether the lane is worth building, not a promise of recovery.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### The named limitation: partial-history clients are silently *included*, not excluded

Exports have no time machine — they start the day someone switches them on, and
`dim_clients.gsc_data_start` records that day per client. So a fixed calendar month is **not the same
window for every client**.

I first assumed such clients would be filtered out by my volume floor. The cell below checks, and
that assumption was wrong in the more dangerous direction: clients whose history begins partway
through March are *inside* the slice. Their "31-day" totals cover a few days, so their impressions
are under-counted, their momentum feature compares a half-window against almost nothing, and their
CTR gap is measured on a fraction of the evidence their peers have — while sitting in the same tier
baseline and being ranked on the same scale.

**The ML-05 fix, chosen now rather than after a result looks good:** either require
`gsc_data_start <= 2026-03-01`, or move to per-client windows as the data skill recommends.

### The other three

- **This data can never tell me *why* a CTR gap exists.** Titles, meta descriptions, URLs and raw
  queries were dropped for privacy, and SERP layout (ads, AI overviews, snippets) was never in it. A
  flagged page is *observed to convert visibility into clicks worse than position-matched peers* —
  never *has a weak title*. The diagnosis stays human.
- **There is no counterfactual.** I never see the same page both reviewed and not reviewed, so
  nothing here supports a claim that fixing a flagged page *will* close its gap. That needs a
  designed experiment.
- **Recent days are still moving.** The nightly job re-merges a rolling five-day window because
  exports arrive late and get revised, so the newest days are the least settled — a second reason to
  develop on a mid-panel month rather than the edge of the panel.

### And one honest limit of the proxy itself

`ctr_gap_pp` is a rule I wrote, so "underperforming" is defined by my tier thresholds and my volume
floor, not by anything the world confirmed. A page can top my queue and be performing exactly as it
should — a navigational page, or one whose peers are a bad comparison set. Until a forward-looking
outcome exists (*did CTR improve after a page was flagged?*), every number here describes an
**authored ranking**, and true Precision@K is not yet available to me.

In [96]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- Measure the named limitation ------------------------------------------
clients = con.sql(f"""
    SELECT client_hash_id, is_active, access_profile, has_gsc_access, gsc_data_start
    FROM {T['dim_clients']}
""").df()
clients["gsc_data_start"] = pd.to_datetime(clients["gsc_data_start"], errors="coerce")
win_start = pd.Timestamp(WINDOW_START)

print(f"clients in dim_clients                        : {len(clients):,}")
print(f"  GSC history starts on/before {WINDOW_START}   : {int((clients.gsc_data_start <= win_start).sum())}")
print(f"  starts DURING or after the window            : {int((clients.gsc_data_start > win_start).sum())}")
print(f"  no gsc_data_start recorded (NULL)            : {int(clients.gsc_data_start.isna().sum())}\n")

lane_clients = clients[clients.client_hash_id.isin(set(lane.client_hash_id))]
partial = lane_clients[lane_clients.gsc_data_start > win_start]
print(f"clients that survive into my lane slice        : {len(lane_clients)} of {len(clients)} "
      f"({len(lane_clients)/len(clients):.1%})")
print(f"  earliest / latest gsc_data_start among them  : "
      f"{str(lane_clients.gsc_data_start.min())[:10]} .. {str(lane_clients.gsc_data_start.max())[:10]}")
print(f"  of those, with PARTIAL March history         : {len(partial)}")

if len(partial):
    rows = lane.client_hash_id.isin(set(partial.client_hash_id))
    print(f"  pages they contribute to the lane            : {int(rows.sum()):,} "
          f"({rows.mean():.1%} of the slice)")
    print(f"  their median window impressions              : {lane[rows].impressions_31d.median():,.0f} "
          f"vs {lane[~rows].impressions_31d.median():,.0f} for full-history clients")
    print("\nVERDICT (observed): partial-history clients are INSIDE the slice, not filtered out by the "
          "volume floor. Their 'monthly' totals cover part of a month, so their features and their "
          "CTR gap rest on less evidence than their tier peers -- a measurement difference the score "
          "currently reads as a performance difference. Fixed in ML-05 by a start-date requirement "
          "or per-client windows.")
    print(f"     Magnitude, stated honestly: {rows.mean():.1%} of the slice. Real and structurally "
          f"important, immaterial in THIS month -- and a per-month check, not something to assume "
          f"away for April or June.")
else:
    print("\nVERDICT (observed): no partial-history client survives the volume floor in this month. "
          "The limitation is still real for other months and must be re-checked, not assumed away.")

share = (lane.groupby("client_hash_id").size().sort_values(ascending=False) / len(lane))
print(f"\nconcentration: largest client {share.iloc[0]:.1%} of rows, top 3 {share.head(3).sum():.1%} -- "
      f"client-holdout validation is mandatory, and results read as 'held on {len(lane_clients)} "
      f"clients with usable March history', never as a statement about the client base.")

clients in dim_clients                        : 104
  GSC history starts on/before 2026-03-01   : 52
  starts DURING or after the window            : 15
  no gsc_data_start recorded (NULL)            : 37

clients that survive into my lane slice        : 36 of 104 (34.6%)
  earliest / latest gsc_data_start among them  : 2025-01-27 .. 2026-03-27
  of those, with PARTIAL March history         : 3
  pages they contribute to the lane            : 65 (0.1% of the slice)
  their median window impressions              : 976 vs 1,902 for full-history clients

VERDICT (observed): partial-history clients are INSIDE the slice, not filtered out by the volume floor. Their 'monthly' totals cover part of a month, so their features and their CTR gap rest on less evidence than their tier peers -- a measurement difference the score currently reads as a performance difference. Fixed in ML-05 by a start-date requirement or per-client windows.
     Magnitude, stated honestly: 0.1% of the slice. Real and st

In [97]:
# --- Receipts: the numbers this notebook actually measured ------------------
import json, os
receipts = {
    "assignment": "ML-04 - Search Intelligence Data Contract",
    "contract": CONTRACT,
    "verified": {
        "Q1_grain_violations": int(len(q1)),
        "Q2_page_day_rows": int(q2.page_day_rows),
        "Q2_distinct_pages": int(q2.distinct_pages),
        "Q2_distinct_clients": int(q2.distinct_clients),
        "Q2_date_span": [str(q2.first_date)[:10], str(q2.last_date)[:10]],
        "Q3_gsc_true_rows": int(q3.gsc_true),
        "Q3_ga4_true_rows": int(q3.ga4_true),
        "Q3_ga4_nulls_missed_by_equals_false": int(q3.ga4_null),
        "Q3_ga4_client_on_row_off": int(q3.ga4_client_on_row_off) if HAS_CLIENT_FLAGS else None,
        "position_sub_1_rows": int(sanity.pos_below_1) if sanity is not None else None,
        "position_numerator_used": pos_num,
        "position_convention": "gsc_avg_position is ZERO-BASED (= sum/impressions); +1 applied",
        "pages_removed_by_position_floor": int(k3.sum() - k4.sum()),
        "pages_rescued_by_the_plus_one": 452,   # measured pre-fix; 0 removed after
    },
    "lane_slice": {"rows": int(len(lane)), "clients": int(lane.client_hash_id.nunique()),
                   "eligibility": {"min_impressions": MIN_IMPRESSIONS,
                                   "min_active_days": MIN_ACTIVE_DAYS,
                                   "min_position": MIN_POSITION}},
    "features": list(FEATURES),
    "proxy": "ctr_gap_pp (leave-one-out, volume-weighted tier baseline) - AUTHORED, not ground truth",
    "leak_experiment": {"leaked_column": "clicks_31d", "leaked_spearman": round(float(rho_l), 3),
                        "honest_spearman": round(float(rho_h), 3), "status": "removed"},
    "honest_baseline": HONEST_BASELINE,
    "open_questions": ["CTR level is ~0.3% across ALL tiers vs ~2.78% documented, and top_3 sits "
                       "below page_1 -- systemic, not an off-by-one",
                       "ga4_data_available FALSE conflates pre-start rows with zero-traffic rows",
                       "partial-history clients inside a fixed calendar window"],
}
print(f"lane: {len(lane):,} pages / {lane.client_hash_id.nunique()} clients | grain violations "
      f"{len(q1)} | span {str(q2.first_date)[:10]}..{str(q2.last_date)[:10]}")
print(f"leak {rho_l:.3f} -> honest {rho_h:.3f} Spearman | R2 {r2_h:.3f} | seed 42")
print(f"open questions: {len(receipts['open_questions'])} (see JSON)")

try:
    os.makedirs("work/outputs", exist_ok=True)
    with open("work/outputs/ml04_data_contract_receipts.json", "w") as f:
        json.dump(receipts, f, indent=2)
    print("\nsaved -> work/outputs/ml04_data_contract_receipts.json")
except Exception as e:
    print(f"\n(receipts not written to disk here: {e})")

lane: 61,881 pages / 36 clients | grain violations 0 | span 2026-03-01..2026-03-31
leak 0.902 -> honest 0.167 Spearman | R2 -0.165 | seed 42
open questions: 3 (see JSON)

saved -> work/outputs/ml04_data_contract_receipts.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### What this contract commits me to next

1. **ML-05** — join `dim_content` only after a key-coverage check, and draw the window diagram for
   `fact_content_query_90d` before touching it.
2. Handle partial-history clients (start-date requirement or per-client windows), and test whether
   `avg_position_31d` can be promoted from context to feature rather than assuming either way.
3. Report **GroupKFold** across folds instead of one grouped split, and try per-client normalisation
   of the gap, which is what the positive-Spearman / negative-R² result points at.
4. Define the forward-looking outcome (*did CTR improve after a page was flagged?*) that turns the
   interim ranking metric into a real Precision@K — June 2026 stays sealed for exactly that.
5. Take the one genuinely open question to the async channel: CTR runs ~0.3% across every tier
   against ~2.78% documented, and `top_3` sits below `page_1` — a level problem the zero-based
   position correction did not explain. (The sub-1 positions themselves are resolved: zero-based.)